# Dashboard trực quan — Valorant Data Mining

Notebook này **không chạy lại mô hình**. Nó đọc các tập dữ liệu đã làm sạch và các bảng kết quả trong `data/processed/` và `outputs/tables/` để trình bày dễ xem hơn.

Trước khi chạy notebook, hãy chạy pipeline một lần tại thư mục dự án:

```bash
make all
```

Sau đó trong Jupyter, chọn **Run All**. Với các phần theo map, chỉ cần đổi giá trị biến `SELECTED_MAP`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 100)

# Notebook nằm trong notebooks/, nên thư mục dự án là thư mục cha.
ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "outputs" / "tables"

print(f"Đang đọc dữ liệu từ: {ROOT}")
assert PROCESSED.exists(), "Không tìm thấy data/processed. Hãy mở notebook từ thư mục dự án."
assert TABLES.exists(), "Không tìm thấy outputs/tables. Hãy chạy `make all` trước."


In [ ]:
# Các dữ liệu đã làm sạch và kết quả khai phá
players = pd.read_csv(TABLES / "player_clusters.csv")
compositions = pd.read_csv(PROCESSED / "team_compositions.csv")
rules = pd.read_csv(TABLES / "agent_rules_by_map.csv")
pick_rates = pd.read_csv(TABLES / "agent_pick_rates_by_map.csv")
exact_comps = pd.read_csv(TABLES / "exact_compositions.csv")
cluster_profiles = pd.read_csv(TABLES / "cluster_profiles_raw.csv")
cluster_eval = pd.read_csv(TABLES / "cluster_evaluation.csv")

print(f"Người chơi đã phân cụm: {len(players):,}")
print(f"Đội hình Agent: {len(compositions):,}")
print(f"Luật kết hợp: {len(rules):,}")
print(f"Map: {', '.join(sorted(compositions['Map'].unique()))}")


## 1. Tổng quan dữ liệu

In [ ]:
n_games = compositions["GameID"].nunique()
n_labeled = compositions["won"].notna().sum()
n_maps = compositions["Map"].nunique()

summary = pd.DataFrame({
    "Chỉ số": ["Số game sau làm sạch", "Số đội hình Agent", "Đội hình có nhãn thắng/thua", "Người chơi phân cụm", "Số map"],
    "Giá trị": [f"{n_games:,}", f"{len(compositions):,}", f"{n_labeled:,}", f"{len(players):,}", n_maps],
})
display(summary)

map_counts = (compositions.groupby("Map")["GameID"].nunique()
              .sort_values(ascending=False)
              .rename("Số game"))
ax = map_counts.plot.bar(figsize=(10, 4), color="#4C78A8")
ax.set_title("Số game sau làm sạch theo map")
ax.set_xlabel("Map")
ax.set_ylabel("Số game")
plt.xticks(rotation=0)
plt.show()


## 2. Phân cụm người chơi

In [ ]:
cluster_summary = (players.groupby(["cluster", "cluster_label"], as_index=False)
                   .agg(
                       players=("PlayerID", "nunique"),
                       win_rate=("win_rate", "mean"),
                       acs=("acs", "mean"),
                       adr=("adr", "mean"),
                       kd=("kd", "mean"),
                       deaths_pr=("deaths_pr", "mean"),
                       assists_pr=("assists_pr", "mean"),
                   )
                   .sort_values("cluster"))

for col in ["win_rate"]:
    cluster_summary[col] = cluster_summary[col].map(lambda x: f"{x:.1%}")
for col in ["acs", "adr", "kd", "deaths_pr", "assists_pr"]:
    cluster_summary[col] = cluster_summary[col].map(lambda x: f"{x:.3f}" if col != "acs" and col != "adr" else f"{x:.1f}")

display(cluster_summary.rename(columns={
    "cluster": "Cụm", "cluster_label": "Tên cụm", "players": "Số người chơi",
    "win_rate": "Win rate", "acs": "ACS", "adr": "ADR", "kd": "K/D",
    "deaths_pr": "Deaths/round", "assists_pr": "Assists/round"
}))


In [ ]:
display(pd.read_csv(ROOT / 'outputs/tables/clustering_summary.csv'))


In [ ]:
# So sánh các chỉ số chính giữa các cụm.
metric_cols = ["kills_pr", "deaths_pr", "assists_pr", "kd", "acs", "adr", "fk_pr", "fd_pr", "multi_pr", "hs"]
metric_labels = {
    "kills_pr": "Kills/R", "deaths_pr": "Deaths/R", "assists_pr": "Assists/R", "kd": "K/D",
    "acs": "ACS", "adr": "ADR", "fk_pr": "First kills/R", "fd_pr": "First deaths/R",
    "multi_pr": "Multi-kill/R", "hs": "HS%"
}

# Chuẩn hóa theo z-score để đặt các chỉ số khác đơn vị lên cùng một biểu đồ.
profile = players.groupby("cluster_label")[metric_cols].mean()
z_profile = (profile - players[metric_cols].mean()) / players[metric_cols].std()
z_profile = z_profile.rename(columns=metric_labels)

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(z_profile, annot=True, fmt=".2f", center=0, cmap="RdBu_r", ax=ax)
ax.set_title("Hồ sơ cụm theo z-score — đỏ: cao hơn trung bình, xanh: thấp hơn")
ax.set_xlabel("Chỉ số")
ax.set_ylabel("Cụm")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


## 3. Agent theo map và luật kết hợp

In [ ]:
# Đổi map tại đây để xem phần phân tích chi tiết.
SELECTED_MAP = "Icebox"
assert SELECTED_MAP in set(compositions["Map"]), f"Map không hợp lệ: {SELECTED_MAP}"
print(f"Map đang xem: {SELECTED_MAP}")


In [ ]:
map_picks = (pick_rates[pick_rates["map"] == SELECTED_MAP]
             .sort_values("pick_rate", ascending=False)
             .head(12)
             .copy())

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=map_picks, x="pick_rate", y="agent", hue="agent", legend=False, palette="viridis", ax=ax)
ax.set_title(f"Agent được chọn nhiều nhất — {SELECTED_MAP}")
ax.set_xlabel("Tỷ lệ xuất hiện trong đội hình")
ax.set_ylabel("Agent")
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
plt.show()

display(map_picks[["agent", "picks", "teams", "pick_rate"]].assign(
    pick_rate=lambda df: df["pick_rate"].map("{:.1%}".format)
).rename(columns={"agent": "Agent", "picks": "Lượt chọn", "teams": "Số đội hình", "pick_rate": "Tỷ lệ chọn"}))


In [ ]:
# Các luật có lift cao nhất trên map được chọn.
map_rules = (rules[rules["map"] == SELECTED_MAP]
             .sort_values(["lift", "support_count"], ascending=[False, False])
             .head(15)
             .copy())

show_rules = map_rules[["rule", "support_count", "support", "confidence", "lift", "win_rate_with_antecedent", "fisher_q_bh"]]
for col in ["support", "confidence", "win_rate_with_antecedent"]:
    show_rules[col] = show_rules[col].map("{:.1%}".format)
show_rules["lift"] = show_rules["lift"].map("{:.2f}".format)
show_rules["fisher_q_bh"] = show_rules["fisher_q_bh"].map("{:.3f}".format)

display(show_rules.rename(columns={
    "rule": "Luật Agent", "support_count": "Số đội hình", "support": "Support",
    "confidence": "Confidence", "lift": "Lift", "win_rate_with_antecedent": "Win rate khi có vế trái",
    "fisher_q_bh": "q-value (BH)"
}))


**Cách đọc luật:** `A, B → C` có nghĩa là khi đội đã chọn A và B thì C thường cũng xuất hiện. Lift lớn hơn 1 cho thấy tổ hợp xuất hiện nhiều hơn kỳ vọng nếu các Agent độc lập. Đây là **đồng xuất hiện**, không phải quan hệ nguyên nhân-kết quả.

## 4. Đội hình Agent và tỷ lệ thắng

In [ ]:
# Các đội hình phổ biến nhất trên map được chọn.
map_comps = (exact_comps[exact_comps["map"] == SELECTED_MAP]
             .sort_values("n", ascending=False)
             .head(15)
             .copy())

fig, ax = plt.subplots(figsize=(12, 6))
plot_data = map_comps.sort_values("win_rate")
ax.errorbar(
    plot_data["win_rate"], plot_data["agents_str"],
    xerr=[plot_data["win_rate"] - plot_data["win_rate_ci_low"],
          plot_data["win_rate_ci_high"] - plot_data["win_rate"]],
    fmt="o", color="#E45756", ecolor="#777777", capsize=3,
)
ax.axvline(0.5, color="black", linestyle="--", linewidth=1, label="50%")
ax.set_title(f"Win rate và khoảng tin cậy 95% — đội hình phổ biến ở {SELECTED_MAP}")
ax.set_xlabel("Win rate")
ax.set_ylabel("Đội hình Agent")
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
show_comps = map_comps[["agents_str", "n", "wins", "win_rate", "win_rate_ci_low", "win_rate_ci_high", "fisher_q_bh", "significant_fdr"]].copy()
for col in ["win_rate", "win_rate_ci_low", "win_rate_ci_high"]:
    show_comps[col] = show_comps[col].map("{:.1%}".format)
show_comps["fisher_q_bh"] = show_comps["fisher_q_bh"].map("{:.3f}".format)

display(show_comps.rename(columns={
    "agents_str": "Đội hình", "n": "Số lần xuất hiện", "wins": "Số trận thắng",
    "win_rate": "Win rate", "win_rate_ci_low": "CI thấp", "win_rate_ci_high": "CI cao",
    "fisher_q_bh": "q-value (BH)", "significant_fdr": "Có ý nghĩa sau FDR"
}))


## 5. Lưu ý khi diễn giải

- Các kết quả phản ánh dữ liệu Valorant chuyên nghiệp từ 2020 đến đầu 2022, không phải meta hiện tại.
- Phân cụm mô tả nhóm hiệu suất trong dữ liệu; không khẳng định đây là vai trò chính thức của người chơi.
- Luật kết hợp mô tả Agent đi cùng nhau; lift cao không chứng minh đội hình đó mạnh hơn.
- Win rate quan sát có thể chịu ảnh hưởng bởi sức mạnh đội tuyển, đối thủ, giải đấu và patch.

## 6. Dự đoán ngoài thời gian và độ tin cậy

Trang kết quả cũ cho biết **điều gì phổ biến**. Phần này kiểm tra **điều gì còn dự đoán được ở giai đoạn sau**. Mỗi ván chỉ xuất hiện một lần; đội đích chọn theo thứ tự chữ cái của mã đội để tránh thứ tự `Team1` vốn lệch về đội thắng. Đọc `predictor_split.json` để xác nhận train/test không giao ngày hoặc MatchID.


In [ ]:
import json
comparison = pd.read_csv(TABLES / "predictor_comparison.csv")
split = json.loads((TABLES / "predictor_split.json").read_text(encoding="utf-8"))
by_map = pd.read_csv(TABLES / "predictor_by_map.csv")
by_month = pd.read_csv(TABLES / "predictor_by_month.csv")
calibration = pd.read_csv(TABLES / "predictor_calibration.csv")

display(Markdown(f"**Dữ liệu:** {split['n_train']:,} ván train đến {split['train_last']}; "
                 f"{split['n_test']:,} ván test từ {split['test_first']}. "
                 f"Tỷ lệ thắng của Team1 gốc: {split['raw_team1_win_rate']:.1%}; "
                 f"sau chọn đội theo mã: {split['positive_rate_train']:.1%} train, "
                 f"{split['positive_rate_test']:.1%} test. "
                 f"Patch chưa xuất hiện trong train: {split['unseen_patch_test_games']:,} ván test.**"))
cols = ["model", "test_accuracy", "test_roc_auc", "test_log_loss", "test_brier",
        "delta_log_loss_vs_baseline", "delta_ci_low", "delta_ci_high"]
display(comparison[cols].style.format({c: "{:.4f}" for c in cols if c != "model"}))

In [ ]:
# Bootstrap lấy mẫu theo MatchID, giữ các ván cùng trận ở một nhóm.
plot = comparison[comparison.model != "Train prevalence"].iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 4.6))
x = range(len(plot))
ax.barh(list(x), plot["delta_log_loss_vs_baseline"], color="#247ba0", alpha=.8)
for i, row in enumerate(plot.itertuples()):
    ax.plot([row.delta_ci_low, row.delta_ci_high], [i, i], color="#17324d", linewidth=3)
ax.axvline(0, color="#555", linewidth=1)
ax.set_yticks(list(x), plot["model"])
ax.set_xlabel("Mức giảm log loss so với mốc tần suất train; khoảng tin cậy 95%")
ax.set_title("Chênh lệch nhỏ; khoảng tin cậy có thể chạm 0")
plt.tight_layout(); plt.show()

## 7. Sai số thay đổi theo map và tháng

Bảng theo map và đường theo tháng giúp thấy mô hình có ổn định khi meta đổi hay không. Map ít ván cần đọc cùng cột `n_games`; chênh lệch ở nhóm nhỏ dễ dao động.


In [ ]:
map_view = by_map[by_map.model.isin(["Random Forest", "AdaBoost"])].copy()
map_view = map_view.sort_values(["Map", "model"])
display(map_view[["Map", "model", "n_games", "win_rate", "roc_auc", "log_loss", "delta_log_loss_vs_baseline"]]
        .style.format({"win_rate":"{:.1%}", "roc_auc":"{:.3f}", "log_loss":"{:.4f}",
                       "delta_log_loss_vs_baseline":"{:+.4f}"}))

fig, ax = plt.subplots(figsize=(10, 4.6))
for model, group in by_month[by_month.model.isin(["Random Forest", "AdaBoost"])].groupby("model"):
    ax.plot(group["month"], group["delta_log_loss_vs_baseline"], marker="o", label=model)
ax.axhline(0, color="#555", linewidth=1)
ax.set(ylabel="Mức giảm log loss so với mốc train", xlabel="Tháng trong tập test",
       title="Độ ổn định dự đoán khi chuyển sang tháng mới")
ax.legend(); plt.tight_layout(); plt.show()

## 8. Hiệu chỉnh xác suất

Điểm dự đoán 60% chỉ đáng tin nếu trong các ván tương tự có xấp xỉ 60% thắng. Mỗi điểm là một nhóm phân vị gồm khoảng 250 ván test; đường chéo là hiệu chỉnh lý tưởng. Đây là kiểm tra mô tả, không dùng để chỉnh mô hình trên test.


In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 6.2))
for model, group in calibration[calibration.model.isin(["Random Forest", "AdaBoost"])].groupby("model"):
    ax.plot(group["mean_predicted"], group["observed_win_rate"], marker="o", label=model)
ax.plot([0, 1], [0, 1], color="#555", linestyle="--", label="Hiệu chỉnh lý tưởng")
ax.set(xlim=(.25, .75), ylim=(.25, .75), xlabel="Xác suất thắng dự đoán trung bình",
       ylabel="Tỷ lệ thắng quan sát", title="Hiệu chỉnh trên tập test")
ax.legend(); plt.tight_layout(); plt.show()

**Kết luận cần giữ:** AUC quanh 0,54 chỉ cao hơn ngẫu nhiên ít. Khoảng tin cậy bootstrap của mức cải thiện log loss phải được xem trước khi khẳng định mô hình tốt hơn mốc. Các bảng theo map/tháng là phân tích lỗi và độ bền, chưa chứng minh Agent nào tạo ra chiến thắng.
